# ЛР3. Цифровой двойник, этап 3: система управления

In [ ]:
# Подготовка среды: папки scripts, autograder и detective должны лежать рядом с notebooks
# (в Colab: загрузите архив материалов дисциплины и распакуйте его в /content)
import sys, os, numpy as np, matplotlib.pyplot as plt
for p in ("../scripts", "scripts", "/content/scripts", "../detective", "/content/detective"):
    if os.path.isdir(p): sys.path.insert(0, os.path.abspath(p))
from srm_model import SRM, simulate_time, cycle_angle_domain
mot = SRM(); deg = np.deg2rad
print("Модель загружена: ВИД", f"{mot.Ns}/{mot.Nr}", "Udc =", mot.Udc, "В")

In [ ]:
rpm, Ts = 600, 1e-4
kw = dict(dt=2e-6, t_end=60/rpm/mot.Nr, Ts=Ts)
res = {c: simulate_time(mot, rpm, deg(10), deg(150), 15, controller=c, **kw) for c in ("hyst", "mpc")}
for c, r in res.items():
    plt.plot(r['t']*1e3, r['i'][:,0], label=c)
plt.axhline(15, ls='--', c='k'); plt.legend(); plt.xlabel('мс'); plt.ylabel('А'); plt.grid(alpha=.3)

In [ ]:
import pathlib
r = res["mpc"]                       # ЗАДАНИЕ: сохраните трассу своего регулятора
act = (np.mod(r['t']*r['w']*mot.Nr - deg(10), 2*np.pi) < deg(140))
out = pathlib.Path("results/lab3"); out.mkdir(parents=True, exist_ok=True)
np.savetxt(out/"lab3_trace.csv", np.column_stack([r['t'], r['i'][:,0], np.full(len(act), 15.0), act]),
           delimiter=",", header="t_s,i_A,i_ref_A,active", comments="")

## Онлайн-идентификация поверхности намагничивания

In [ ]:
import srm_adapt, json
k = 1e-3                              # ЗАДАНИЕ: исследуйте несколько значений
a = srm_adapt.run(k=k, cycles=15)
plt.plot(np.arange(1, 16), a['rms'], 'o-'); plt.xlabel('период'); plt.ylabel('СКО тока, А'); plt.grid(alpha=.3)
n1 = int(np.argmax(a['rms'] <= 1.0)) + 1
json.dump({"k": k, "periods_to_1A": n1}, open(out/"lab3_adapt.json", "w"))
print("периодов до СКО <= 1 А:", n1)

## Как звучат пульсации момента
Запустите `sound_of_srm.py` и прослушайте файлы в папке `sound`. Опишите различие спектров и объясните, почему это не расчет акустического шума.

In [ ]:
try:
    from IPython.display import Audio, display
    for f in ("hyst_band0p8A.wav", "mpc_Ts100us.wav"):
        for d in ("../sound", "sound", "/content/sound"):
            if os.path.exists(os.path.join(d, f)): display(Audio(os.path.join(d, f)))
except ImportError:
    print("Прослушайте файлы в папке sound")